# Libera - Earth Target Geolocation Example

This notebook demonstrates how to use the dynamic kernels provided by Libera to compute the geolocation of Earth targets observed by the Libera instrument. We will load the kernels, extract ephemeris and attitude data, compute the geolocation, and visualize the results. This example uses test data available in this repository and shows the operational mode of Libera called "Earth Target".

In [ ]:
import logging
from datetime import datetime
from pathlib import Path
import importlib
import os

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from curryer import utils, spicetime, meta
from curryer.compute import spatial
from curryer import spicierpy as sp

from libera_utils.libera_spice.kernel_manager import KernelManager

DYNAMIC_KERNEL_PATH = Path(os.getcwd()).parent / "tests/test_data/dynamic_kernels"

xr.set_options(display_width=120, display_max_rows=30)
np.set_printoptions(linewidth=120)
utils.enable_logging(log_level=logging.DEBUG, extra_loggers=[__name__])
figsize = (12, 6)

In [ ]:
# Using KernelManager to load kernels and furnish them to SpiceyPy
# This will load static kernels, NAIF kernels, and Libera dynamic kernels.
# This allows SpiceyPy or SpicierPy calls (from curryer) to access them and inspect them.

km = KernelManager()
km.load_static_kernels()
km.load_naif_kernels()
km.load_libera_dynamic_kernels(DYNAMIC_KERNEL_PATH)

# Inspect Satellite Position

In [ ]:
utc_range = np.asarray(['2028-01-02 00:13:47', '2028-01-02 00:29:12'], dtype='M8[us]')
et_range = spicetime.adapt(utc_range, 'iso', 'et')
et_times = np.arange(*et_range, .01)[2:-2]
dt64_times = spicetime.adapt(et_times, 'et', 'dt64')

In [ ]:
positions = []
for sample_et in et_times:
    sample_arr, _ = sp.spkezr(
        sp.obj.Body('JPSS4_SC').id,
        sample_et,
        ref='ITRF93',
        abcorr='NONE',
        obs=sp.obj.Body('EARTH').id,
    )
    positions.append(sample_arr)
positions = np.vstack(positions)
velocities = positions[:, 3:]
positions = positions[:, :3]

## Inspect Spacecraft Telemetry

In [ ]:
figsize = (12, 6)
fig, ax = plt.subplots(figsize=figsize)

for ith, field in enumerate(['X', 'Y', 'Z']):
    ax.scatter(dt64_times, positions[:, ith], s=1, label=field)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Ephemeris (ECEF)');

In [ ]:
jpss4_lla = spatial.ecef_to_geodetic(positions, degrees=True, meters=False)

fig, ax = plt.subplots(figsize=figsize)

ax.scatter(dt64_times, jpss4_lla[:,0], s=1, label="lon")
ax.scatter(dt64_times, jpss4_lla[:,1], s=1, label="lat")
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Ephemeris (Ellipsoid)');

# Inspect AzEl

In [ ]:
azrot = []
for sample_et in et_times:
    sample_arr = sp.pxform('LIBERA_AZ_COORD', 'LIBERA_BASE_COORD', sample_et)
    sample_arr = sp.m2eul(sample_arr, 3, 2, 1)
    # Why need the negative?
    sample_arr = np.rad2deg(-sample_arr[0])
    azrot.append(sample_arr)
azrot = np.vstack(azrot)

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

ax.scatter(dt64_times, azrot, s=1, label='Az')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Az Rotation (deg)');

In [ ]:
elrot = []
for sample_et in et_times:
    sample_arr = sp.pxform('LIBERA_EL_COORD', 'LIBERA_AZ_COORD', sample_et)
    sample_arr = sp.m2eul(sample_arr, 3, 2, 1)
    sample_arr = np.rad2deg(sample_arr[2])
    elrot.append(-sample_arr)
elrot = np.vstack(elrot)

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

ax.scatter(dt64_times, elrot, s=1, label='El')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - El Rotation (deg)');

# Geolocation Inspection

In [ ]:
ugps_times = spicetime.adapt(pd.date_range(dt64_times[1], dt64_times[-2], freq="10ms", inclusive="left"), "iso")
ugps_times = spicetime.adapt(pd.date_range("2028-01-02T00:13:47", "2028-01-02T00:29:12", freq="10ms", inclusive="left"), "iso")

In [ ]:
#ugps_times = spicetime.adapt(pd.date_range(dt64_times[2], dt64_times[-2], freq="10ms", inclusive="left"), "iso")
ellips_lla_df, sc_xyz_df, ellips_qf_ds = spatial.instrument_intersect_ellipsoid(
            ugps_times, sp.obj.Body("LIBERA_SW_RAD", frame=True), geodetic=True, degrees=True
        )

In [ ]:
from curryer.compute.constants import SpatialQualityFlags as SQF
qf_counts = ellips_qf_ds.value_counts()
print(qf_counts)
#per_good = qf_counts[0] / qf_counts.sum()
#print(per_good)

slc = slice(*spicetime.adapt(["2028-01-02 00:21:22", "2028-01-02 00:21:36"], "iso"))
print("Mean Lat: ", np.nanmean(ellips_lla_df.loc[slc]["lat"]))
print("Mean Lon: ", np.nanmean(ellips_lla_df.loc[slc]["lon"]))

In [ ]:
dt64_times = pd.to_datetime(spicetime.adapt(ellips_lla_df.index, to='dt64'))
lons = ellips_lla_df['lon'].values
lats = ellips_lla_df['lat'].values

dt64_times[0], dt64_times[-1]

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(figsize[0], figsize[1] * 2))

idx = Ellipsis

ax1.scatter(dt64_times[idx], lons[idx], s=1, label='Lon')
ax1.scatter(dt64_times[idx], lats[idx], s=1, label='Lat')
ax1.set_xlabel('Time (UTC)')
ax1.set_ylabel('Degrees')

ax2.scatter(lons[idx], lats[idx], c=dt64_times[idx], s=1, alpha=0.1)
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
for ax in (ax1, ax2):
    ax.legend() ; ax.grid()
    ax.set_title('Libera - Geolocation (ECEF)');

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(figsize[0], figsize[1] * 2))

idx = Ellipsis
# idx = slice(None, None, 10)
# slc = slice(*pd.to_datetime(['2028-01-02 00:21', '2028-01-02 00:22']))
# idx = np.where((dt64_times >= slc.start) & (dt64_times < slc.stop))

ax.scatter(lons[idx], lats[idx], c=dt64_times[idx], s=2, alpha=0.2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# ax.set_ylim(25, 32)
# ax.set_xlim(20, 27)

for ax in (ax,):
    ax.grid()  # ; ax.legend()
    ax.set_title('Libera - Geolocation (Ellipsoid)');

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(figsize[0], figsize[1] * 2))

idx = Ellipsis
# idx = slice(None, None, 10)
# slc = slice(*pd.to_datetime(['2028-01-02 00:21', '2028-01-02 00:22']))
# # slc = slice(*pd.to_datetime(['2028-01-02 00:21:22', '2028-01-02 00:21:36']))
# idx = np.where((dt64_times >= slc.start) & (dt64_times < slc.stop))

ax.scatter(lons[idx], lats[idx], c=dt64_times[idx], s=2, alpha=0.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

ax.scatter(23.39, 28.55, c='red', s=200, alpha=0.5, marker='*', label='Focus')

ax.set_ylim(25, 32)
ax.set_xlim(20, 27)

for ax in (ax, ):
    ax.grid() ; ax.legend()
    ax.set_title('Libera - Geolocation (Ellipsoid)');